In [1]:
# just show some conformers
smiles = "CC1=C(CCC(=O)N1)C2=CC(=C(N2)C3=CC(=C(C(=N3)C4=CC(=C(C(=N4)C(=O)O)CCC(=O)O)C)C)CCC(=O)O)C"
# smiles = 'C1CCCCC1'
import py3Dmol
from rdkit import Chem
from rdkit.Chem import AllChem

mol = Chem.AddHs(Chem.MolFromSmiles(smiles))
AllChem.EmbedMultipleConfs(mol, numConfs=10)

view = py3Dmol.view(width=300, height=300)
for conf_id in range(3):  # Show the first 3 conformers
    mb = Chem.MolToMolBlock(mol, confId=conf_id)
    view.addModel(mb, "mol")
    view.setStyle({"stick": {}})
    view.zoomTo()
view.show()

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [2]:
# rdkit doesn't work

ref_conf = 0  # use first conformer as reference
for conf_id in range(1, mol.GetNumConformers()):
    AllChem.AlignMol(mol, mol, prbCid=conf_id, refCid=ref_conf)

for conf_id in range(3):  # Show the first 3 conformers
    mb = Chem.MolToMolBlock(mol, confId=conf_id)
    view.addModel(mb, "mol")
    view.setStyle({"stick": {}})
    view.zoomTo()
view.show()

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [ ]:
# align using rdkit
import numpy as np
import py3Dmol
from rdkit import Chem
from rdkit.Chem import AllChem, GetPeriodicTable
from scipy.spatial.transform import Rotation

from rapiat.geometry import (
    get_conformer_positions,
    set_conformer_positions,
)

mol = Chem.AddHs(Chem.MolFromSmiles("C1CCCCC1"))
AllChem.EmbedMultipleConfs(mol, numConfs=10)

# Get atomic masses
masses = np.array(
    [GetPeriodicTable().GetAtomicWeight(atom.GetAtomicNum()) for atom in mol.GetAtoms()]
)  # shape: (num_atoms,)

positions = get_conformer_positions(mol)
# Example: align conformer 1 to reference conformer 0
ref_conf = 0
for i in range(mol.GetNumConformers()):
    rot, rmsd = Rotation.align_vectors(positions[ref_conf], positions[i])
    positions[i] = rot.apply(positions[i])
    print("RMSD:", rmsd, np.linalg.norm(positions[i] - positions[ref_conf]))

set_conformer_positions(mol, positions)

view = py3Dmol.view(width=300, height=300)
for conf_id in range(3):  # Show the first 3 conformers
    mb = Chem.MolToMolBlock(mol, confId=conf_id)
    view.addModel(mb, "mol")
    view.setStyle({"stick": {}})
    view.zoomTo()
view.show()